# Initializing MinIO and PostgreSQL

In [163]:
import os, sys

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import List as mapping_list
import psycopg2
from io import BytesIO
from minio import Minio
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())


True

In [107]:
minio_client = Minio(
    "localhost:9100",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

bucket_name = 'pulse-bucket-1'

In [108]:
spark = SparkSession.builder.appName("NormalizeData")\
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9100") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

#  Function To load all files from minio 

In [156]:
import os
import tempfile
import pandas as pd
from io import BytesIO
from pyspark.sql import SparkSession
import re

def load_all_files_from_minio(minio_client, bucket_name, spark):
    # Dictionary to store all dataframes
    dataframes = {}
    
    # List all objects in the bucket
    objects = minio_client.list_objects(bucket_name, recursive=True)
    print("Listing available files in the bucket...")
    
    # Process each object
    for obj in objects:
        file_name = obj.object_name
        print(f"Processing file: {file_name}")
        
        # Skip if not a recognized data file
        if not (file_name.endswith('.csv') or file_name.endswith('.xlsx') or 
                file_name.endswith('.parquet') or file_name.endswith('.json')):
            print(f"Skipping {file_name} - unsupported format")
            continue
            
        try:
            # Extract data type from filename using regex
            match = re.search(r'messy_([a-zA-Z]+)_data', file_name)
            if match:
                data_type = match.group(1)
            else:
                match = re.search(r'messy_([a-zA-Z]+)_([a-zA-Z]+)_data', file_name)
                if match:
                    data_type = f"{match.group(1)}_{match.group(2)}"
                else:
                    data_type = os.path.splitext(os.path.basename(file_name))[0]
            # Load the file
            df = load_file_from_minio(minio_client, bucket_name, file_name)
            
            # Store in the dictionary with appropriate name
            df_name = f"{data_type}_df"
            dataframes[df_name] = df
            print(f"Successfully loaded {file_name} as {df_name}")
            
        except Exception as e:
            print(f"Error processing {file_name}: {str(e)}")
    
    print(f"Loaded {len(dataframes)} dataframes: {', '.join(dataframes.keys())}")
    return dataframes

import uuid

def load_file_from_minio(minio_client, bucket_name, file_name):

    obj = minio_client.get_object(bucket_name, file_name)
    data = obj.read()
    obj.close()
    obj.release_conn()

    if file_name.endswith(".csv"):
        pdf = pd.read_csv(BytesIO(data))
    elif file_name.endswith(".xlsx"):
        pdf = pd.read_excel(BytesIO(data))
    elif file_name.endswith(".parquet"):
        pdf = pd.read_parquet(BytesIO(data))
    elif file_name.endswith(".json"):
        pdf = pd.read_json(BytesIO(data))
    else:
        raise ValueError(f"Unsupported file format: {file_name}")

    # Use a unique temp path and materialize before deleting
    temp_csv = os.path.join(
        tempfile.gettempdir(),
        f"temp_{uuid.uuid4().hex}_{os.path.basename(file_name)}.csv",
    )
    pdf.to_csv(temp_csv, index=False)

    spark_df = spark.read.csv(temp_csv, header=True, inferSchema=True).cache()
    _ = spark_df.count()  # materialize so Spark no longer needs the file

    try:
        os.remove(temp_csv)
    except Exception:
        pass

    return spark_df


In [157]:
all_dataframes = load_all_files_from_minio(minio_client, bucket_name, spark)
 
customer_df = all_dataframes.get('customer_df')


Listing available files in the bucket...
Processing file: messy_address_data.xlsx
Successfully loaded messy_address_data.xlsx as address_df
Processing file: messy_category_data.xlsx
Successfully loaded messy_category_data.xlsx as category_df
Processing file: messy_customer_data.xlsx
Successfully loaded messy_customer_data.xlsx as customer_df
Processing file: messy_customer_sessions_data.xlsx
Successfully loaded messy_customer_sessions_data.xlsx as customer_sessions_df
Processing file: messy_inventory_data.xlsx
Successfully loaded messy_inventory_data.xlsx as inventory_df
Processing file: messy_marketing_campaigns_data.xlsx
Successfully loaded messy_marketing_campaigns_data.xlsx as marketing_campaigns_df
Processing file: messy_order_items_data.xlsx
Successfully loaded messy_order_items_data.xlsx as order_items_df
Processing file: messy_orders_data.xlsx
Successfully loaded messy_orders_data.xlsx as orders_df
Processing file: messy_payments_data.xlsx
Successfully loaded messy_payments_dat

In [158]:
customer_df.show(5)

+-------+------------------+-----------------+------+-------------------+--------------------+--------+------------------+----------------+--------------------+--------------------+-----------------+-------------------+---------------+--------------+------+--------------+-------+
|cust_id|         full_name|customer_category|   sex|         birth_date|     account_created|  status|acquisition_source|   customer_tier|   created_timestamp|       email_address|     phone_number| last_purchase_date|total_purchases|loyalty_points|   age|lifetime_value|country|
+-------+------------------+-----------------+------+-------------------+--------------------+--------+------------------+----------------+--------------------+--------------------+-----------------+-------------------+---------------+--------------+------+--------------+-------+
|  10369|   Cynthia Gregory|              B2C|  Male|1971-10-04 00:00:00|2022-11-11 11:16:...|Inactive|          LinkedIn|      High Value|2024-12-21 08:06:.

# Function to get a single file from  MinIO


import tempfile

def load_file_from_minio(file_name):
    obj = minio_client.get_object(bucket_name, file_name)
    data = obj.read()
    obj.close()
    obj.release_conn()

    if file_name.endswith(".csv"):
        df = pd.read_csv(BytesIO(data))
    elif file_name.endswith(".xlsx"):
        df = pd.read_excel(BytesIO(data))
    elif file_name.endswith(".parquet"):
        df = pd.read_parquet(BytesIO(data))
    elif file_name.endswith(".json"):
        df = pd.read_json(BytesIO(data))
    else:
        raise ValueError("Unsupported file format")

    # Use tempfile.gettempdir() to get the system's temporary directory
    temp_csv = os.path.join(tempfile.gettempdir(), "temp.csv")
    df.to_csv(temp_csv, index=False)

    spark_df = spark.read.csv(temp_csv, header=True, inferSchema=True)
    return spark_df

In [164]:
conn = psycopg2.connect(
    host="localhost",
    database=os.getenv("POSTGRES_DATABASE_NAME"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

In [165]:
cur = conn.cursor()
cur.execute(
    """
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
"""
)

In [166]:
columns_info = cur.fetchall()
columns_info

[('reviews', 'is_verified_purchase', 'boolean'),
 ('marketing_campaigns', 'start_date', 'date'),
 ('marketing_campaigns', 'end_date', 'date'),
 ('marketing_campaigns', 'budget', 'numeric'),
 ('marketing_campaigns', 'spent_amount', 'numeric'),
 ('marketing_campaigns', 'impressions', 'integer'),
 ('marketing_campaigns', 'clicks', 'integer'),
 ('marketing_campaigns', 'conversions', 'integer'),
 ('marketing_campaigns', 'created_at', 'timestamp without time zone'),
 ('customer_sessions', 'session_start', 'timestamp without time zone'),
 ('customer_sessions', 'session_end', 'timestamp without time zone'),
 ('customer_sessions', 'pages_viewed', 'integer'),
 ('customer_sessions', 'products_viewed', 'integer'),
 ('customer_sessions', 'conversion_flag', 'boolean'),
 ('customer_sessions', 'cart_abandonment_flag', 'boolean'),
 ('categories', 'is_active', 'boolean'),
 ('categories', 'created_at', 'timestamp without time zone'),
 ('customers', 'date_of_birth', 'date'),
 ('customers', 'registration_d

In [117]:
cur.close()
conn.close()

# Mapping with Predefined Lists

In [118]:
import importlib
importlib.reload(mapping_list)

<module 'List' from 'd:\\VS CODE\\pulse\\mapping\\List.py'>

In [119]:
def normalize_dataframe(df, column_variants, mapped_cols):
    variant_to_standard = {
        v.lower(): std_col
        for std_col, variants in column_variants.items()
        for v in variants
    }

    new_columns = []
    for col in df.columns:
        col_lower = col.lower()
        if col_lower in variant_to_standard:
            std_col = variant_to_standard[col_lower]
            new_columns.append(std_col)
            mapped_cols[std_col] = col
        else:
            new_columns.append(col)

    for old_col, new_col in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old_col, new_col)

    missing_cols = []
    for std_col in column_variants.keys():
        if std_col not in df.columns:
            df = df.withColumn(std_col, lit(None))
            missing_cols.append(std_col)

    schema_cols = list(column_variants.keys())
    extra_cols = [c for c in df.columns if c not in schema_cols]

    new_df = df.select(schema_cols)
    df_extra = df.select(schema_cols + extra_cols)
    

    return new_df, df_extra, extra_cols, missing_cols, mapped_cols

In [120]:
# base_dir = os.getcwd()
# file_path_excel = os.path.join(base_dir, "./../faker/messy_inventory_data.xlsx")
# file_path_csv = os.path.join(base_dir, "./../faker/messy_inventory_data.csv")

In [121]:
# excel_df = pd.read_excel(file_path_excel, engine="openpyxl")
# excel_df.to_csv(file_path_csv, index=False)

In [122]:
# df = spark.read.csv(file_path_csv, header=True, inferSchema=True)

In [123]:
# df.show(5)

# Implementing RapidFuzz

In [124]:
from rapidfuzz import fuzz, process


def rapidfuzz_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=85):
    """
    Map columns using RapidFuzz's string matching algorithms
    Returns normalized dataframe, remaining missing columns, extra columns, and mapped columns
    """
    for missing_col in missing_cols[:]:
        # Use process.extractOne to find the best match
        match = process.extractOne(
            missing_col,
            extra_cols,
            scorer=fuzz.ratio,  # Can also use fuzz.WRatio or fuzz.token_sort_ratio
            score_cutoff=threshold,
        )

        if match:  # match will be (matched_string, score, index)
            best_match, score = match[0], match[1]
            print(f"Mapping: {best_match} -> {missing_col}: {score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    # Rename columns based on mapping
    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)

    return df, missing_cols, extra_cols, mapped_cols



# Implementing NLTK

In [125]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.metrics.distance import edit_distance
from difflib import SequenceMatcher
import re

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [126]:
def preprocess_column_name(column):
    """Split camelCase and snake_case, convert to lowercase"""
    # Split by underscore and remove special characters
    words = "".join(c if c.isalnum() else " " for c in column).split()
    # Split camelCase
    result = []
    for word in words:
        result.extend(filter(None, re.split("([A-Z][a-z]*)", word)))
    return [w.lower() for w in result if w]

#### Implementing Jaccard Similarity

In [127]:
def jaccard_similarity(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    intersection = len(set(source_col).intersection(set(target_col)))
    union = len(set(source_col).union(set(target_col)))
    jaccard_similarity = intersection / union if union > 0 else 0
    return jaccard_similarity

#### Implementing Sequence Matcher

In [128]:
def sequence_matching(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    return SequenceMatcher(None, source_col, target_col).ratio()

#### Implementing Edit Distance

In [129]:
def editing_distance(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    max_len = max(len(source_col), len(target_col))
    return 1 - (edit_distance(source_col, target_col) / max_len)

#### Combining them all

In [130]:
def mapping_with_combination(df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            final = (
                0.4 * jaccard_similarity(missing_col, extra_col)
                + 0.3 * sequence_matching(missing_col, extra_col)
                + 0.3 * editing_distance(missing_col, extra_col)
            )
            print(f"{missing_col} -> {extra_col}: {final:.2f}")
            if final > best_score:
                best_score = final
                best_match = extra_col

        if best_match and best_score > threshold:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)
    return df, missing_cols, extra_cols, mapped_cols




#### Wordnet Mapping

In [131]:
# def preprocess_column_name(column):
#     name = re.sub("([A-Z][a-z]+)", r" \1", column)
#     name = re.sub("_", " ", name)
#     tokens = word_tokenize(name.lower())
#     return [token for token in tokens if token.isalpha()]

In [132]:
def get_wordnet_synsets(word):
    return wordnet.synsets(word)

In [133]:
def calculate_semantic_similarity(missing_col, extra_col):
    missing_tokens = preprocess_column_name(missing_col)
    extra_tokens = preprocess_column_name(extra_col)
    if not missing_tokens or not extra_tokens:
        return 0.0
    max_similarities = []
    for token1 in missing_tokens:
        synsets1 = get_wordnet_synsets(token1)
        if not synsets1:
            continue
        token_similarities = []
        for token2 in extra_tokens:
            synsets2 = get_wordnet_synsets(token2)
            if not synsets2:
                continue
            similarities = [
                s1.path_similarity(s2)
                for s1 in synsets1
                for s2 in synsets2
                if s1.path_similarity(s2) is not None
            ]
            if similarities:
                token_similarities.append(max(similarities))
        if token_similarities:
            max_similarities.append(max(token_similarities))
    return sum(max_similarities) / len(max_similarities) if max_similarities else 0.0

In [134]:
def semantic_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.6):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            similarity = calculate_semantic_similarity(missing_col, extra_col)
            if similarity > best_score:
                best_score = similarity
                best_match = extra_col
        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)
        for new_col, old_col in mapped_cols.items():
            df = df.withColumnRenamed(old_col, new_col)
    return df, missing_cols, extra_cols, mapped_cols


# Implementing ydata-profiling

In [135]:
# from ydata_profiling import ProfileReport
# pdf = df.toPandas()

# profile = ProfileReport(pdf, title="Pandas Profiling Report", explorative=True)
# profile.to_file("pandas_profiling_report.html")
# desc = profile.description_set

# def pandas_profiling_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.87)
#     for missing_col in missing_cols[:]:
#         best_match = None
#         best_score = 0

#         for extra_col in extra_cols[:]:
#             corr = abs(pdf[missing_col].corr(pdf[extra_col]))
#             if corr > best_score:
#                 best_score = corr
#                 best_match = extra_col

#         if best_match and best_score > threshold:
#             print(
#                 f"Data-based Mapping: {best_match} -> {missing_col} (corr={best_score:.2f})"
#             )
#             mapped_cols[missing_col] = best_match
#             pdf = pdf.rename(columns={best_match: missing_col})
#             extra_cols.remove(best_match)
#             missing_cols.remove(missing_col)
        
#     new_df = spark.createDataFrame(pdf)
#     return new_df, missing_cols, extra_cols, mapped_cols
# new_df, missing, extra_cols, mapped = pandas_profiling_mapping(
#     df, missing, extra_cols, mapped, threshold=0.87
# )

# print("\nNormalized DataFrame:")
# new_df.show(5)

# print("\nMissing columns:")
# print(missing)

# print("\nExtra columns:")
# print(extra_cols)

# print("\nMapped columns:")
# print(mapped)

# Implementing spaCy

In [136]:
import spacy

def spacy_column_mapping(
    df,
    missing_cols,
    extra_cols,
    mapped_cols,
    threshold=0.87
):
    nlp = spacy.load("en_core_web_md")

    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        missing_doc = nlp(" ".join(preprocess_column_name(missing_col)))

        for extra_col in extra_cols[:]:
            extra_doc = nlp(" ".join(preprocess_column_name(extra_col)))
            similarity = missing_doc.similarity(extra_doc)

            if similarity > best_score:
                best_score = similarity
                best_match = extra_col

        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)

    return df, missing_cols, extra_cols, mapped_cols



# Implementing Word2Vec

In [ ]:
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
import re


def load_word2vec_model(df , extra_df):
    # Get all unique column names from both dataframes
    all_columns = list(set(df.columns + extra_df.columns))
    
    # Load pre-trained Word2Vec model - you can use different pre-trained models
    try:
        # Try loading Google's pre-trained model (you need to download this separately)
        model = KeyedVectors.load_word2vec_format(
            "GoogleNews-vectors-negative300.bin", binary=True
        )
    except:
        # Fallback to training a simple model on your column names
        # This is just a basic fallback - ideally you should use a pre-trained model
        sentences = [preprocess_column_name(col) for col in all_columns]
        model = Word2Vec(sentences, vector_size=100, window=5, min_count=1)
        model = model.wv
    return model


def calculate_word2vec_similarity(col1, col2, model):
    words1 = preprocess_column_name(col1)
    words2 = preprocess_column_name(col2)

    if not words1 or not words2:
        return 0.0
    vec1 = []
    vec2 = []

    for word in words1:
        try:
            vec1.append(model[word])
        except KeyError:
            continue

    for word in words2:
        try:
            vec2.append(model[word])
        except KeyError:
            continue

    if not vec1 or not vec2:
        return 0.0

    vec1_avg = np.mean(vec1, axis=0)
    vec2_avg = np.mean(vec2, axis=0)

    similarity = np.dot(vec1_avg, vec2_avg) / (
        np.linalg.norm(vec1_avg) * np.linalg.norm(vec2_avg)
    )
    return float(similarity)


def word2vec_column_mapping(df, extra_df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    model = load_word2vec_model(df , extra_df)

    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold

        for extra_col in extra_cols[:]:
            similarity = calculate_word2vec_similarity(missing_col, extra_col, model)
            print(f"{missing_col} -> {extra_col}: {similarity:.2f}")
            if similarity > best_score:
                best_score = similarity
                best_match = extra_col

        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

        for new_col, old_col in mapped_cols.items():
            df = df.withColumnRenamed(old_col, new_col)

    return df, missing_cols, extra_cols, mapped_cols



# Implementing Hugging Face RoBERTa Transformer

from sentence_transformers import SentenceTransformer
import torch

roberta = SentenceTransformer("roberta-large-nli-stsb-mean-tokens")

def roberta_similarity(df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    missing_embeddings = roberta.encode(missing_cols, convert_to_tensor=True)
    print(missing_embeddings)
    extra_embeddings = roberta.encode(extra_cols, convert_to_tensor=True)
    print(extra_embeddings)

    for i, missing_col in enumerate(missing_cols):
        sims = util.cos_sim(missing_embeddings[i], extra_embeddings)[0]
        print(sims)
        best_score, best_idx = torch.max(sims, dim=0)
        best_score = best_score.item()
        best_match = extra_cols[best_idx]

        if best_score >= threshold:
            print(f"BERT Mapping: {best_match} -> {missing_col} (score={best_score:.2f})")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)
    for schema_col, df_col in mapped_cols.items():
        df = df.withColumnRenamed(df_col, schema_col)
    return df, missing_cols, extra_cols, mapped_cols



# Implementing GPT-OSS

import json
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN"),
)

def gptoss_schema_mapping(df, missing_cols, extra_cols, mapped_cols):
    # Convert to Pandas for sampling
    pdf = df.limit(5).toPandas()

    schema_info = {
        "missing_cols": missing_cols,
        "extra_cols": extra_cols,
        "already_mapped": mapped_cols,
        "sample_data": pdf.to_dict(orient="list"),
    }

    prompt = f"""
        You are a data engineer helping with schema alignment.

        I have a dataset with extra columns and some missing schema columns.
        Please map extra columns to missing schema columns based on semantics and sample data.

        Input (JSON):
        {json.dumps(schema_info, indent=2)}

        Return ONLY valid JSON with this exact structure:
        {{
        "mapped_cols": {{"schema_col": "df_col", ...}},
        "remaining_missing_cols": [],
        "remaining_extra_cols": []
        }}
    """

    # Call Hugging Face GPT OSS
    response = client.responses.create(
        model="openai/gpt-oss-20b:cerebras",
        input=prompt,
    )

    result = response.output_text.strip()

    # Parse JSON safely
    try:
        model_mapping = json.loads(result)

        # Update mappings
        mapped_cols.update(model_mapping["mapped_cols"])
        missing_cols = model_mapping["remaining_missing_cols"]
        extra_cols = model_mapping["remaining_extra_cols"]

        # Apply renaming in Spark DataFrame
        for schema_col, df_col in model_mapping["mapped_cols"].items():
            df = df.withColumnRenamed(df_col, schema_col)

    except Exception as e:
        print("❌ Error parsing model output:", e)
        model_mapping = {}

    return df, missing_cols, extra_cols, mapped_cols

In [145]:
def mapping(df, column_variants , mapped):
    new_df, extra_df, extra_cols, missing_cols, mapped_cols = normalize_dataframe(
        df, column_variants, mapped
    )
    if missing_cols:
        print("\nAfter Initial Normalization:")
        print(f"Missing columns: {missing_cols}")
        print(f"Using ML Models to find missing columns:")
        new_df, missing_cols, extra_cols, mapped_cols = rapidfuzz_column_mapping(
            df, missing_cols, extra_cols, mapped_cols, threshold=85
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = mapping_with_combination(
            df, missing_cols, extra_cols, mapped_cols, threshold=0.87
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = semantic_column_mapping(
            df, missing_cols, extra_cols, mapped_cols, threshold=0.6
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = spacy_column_mapping(
            df, missing_cols, extra_cols, mapped_cols, threshold=0.87
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = word2vec_column_mapping(
            df, extra_df, missing_cols, extra_cols, mapped_cols
        )

    # if missing_cols:
    #   new_df, missing_cols, extra_cols, mapped_cols = roberta_similarity(
    #        df, missing_cols, extra_cols, mapped_cols, threshold=0.87
    #    )

    # if missing_cols:
    #    new_df, missing_cols, extra_cols, mapped_cols = gptoss_schema_mapping(
    #        df, missing_cols, extra_cols, mapped_cols
    #    )

    return new_df, extra_df, extra_cols, missing_cols, mapped_cols




In [146]:
tables_names =  [table for table,cols, _ in columns_info]
tables_names = set(tables_names)
print(tables_names)

mapped = {col: "" for table, col, _ in columns_info if table == "category"}
print(mapped)

{'suppliers', 'customers', 'payments', 'shopping_cart', 'inventory', 'addresses', 'reviews', 'customer_sessions', 'wishlist', 'orders', 'products', 'marketing_campaigns', 'categories', 'order_items'}
{}


In [147]:
category_df = all_dataframes.get('category_df')
category_df.show(5)

+------+------------------+-------------+-----------+--------------------+--------------------+-------------+--------------------+----------------+---------------+------------------+
|cat_id|          cat_name|parent_cat_id|active_flag|        date_created|       category_desc|product_count|     breadcrumb_path|display_sequence|hierarchy_level|          url_slug|
+------+------------------+-------------+-----------+--------------------+--------------------+-------------+--------------------+----------------+---------------+------------------+
|   295|           Spoorts|          222|       True|2024-08-24 17:55:...|Fuga praesentium ...|          611|                NULL|              27|              0|              NULL|
|   259|Premium Collection|          103|       True|2025-08-27 01:46:...|Explicabo archite...|          390|                NULL|              64|           NULL|premium-collection|
|   171|     Boys Clothing|         NULL|       True|2024-11-02 18:52:...|           

In [148]:
final_df, extra_df, extra_cols, missing_cols, mapped_cols = mapping(
    category_df, mapping_list.mapping_dict_categories, mapped
)

print("\nNormalized DataFrame:")
final_df.show(5)

print("\nDataFrame with Extra Columns:")
extra_df.show(5)

print("\nMissing columns:")
print(missing_cols)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped_cols)


After Initial Normalization:
Missing columns: ['created_at']
Using ML Models to find missing columns:
created_at -> date_created: 0.28
created_at -> category_desc: 0.00
created_at -> product_count: 0.00
created_at -> breadcrumb_path: 0.00
created_at -> display_sequence: 0.00
created_at -> hierarchy_level: 0.00
created_at -> url_slug: 0.00
created_at -> date_created: 0.54
created_at -> category_desc: -0.06
created_at -> product_count: -0.14
created_at -> breadcrumb_path: -0.08
created_at -> display_sequence: 0.14
created_at -> hierarchy_level: 0.08
created_at -> url_slug: -0.03

Normalized DataFrame:
+-----------+------------------+------------------+---------+--------------------+--------------------+-------------+--------------------+----------------+---------------+------------------+
|category_id|     category_name|parent_category_id|is_active|        date_created|       category_desc|product_count|     breadcrumb_path|display_sequence|hierarchy_level|          url_slug|
+---------

In [162]:
def process_all_dataframes(all_dataframes, columns_info, mapping_list):
    # Dictionary to map dataframe names to PostgreSQL table names
    df_to_table = {
        'customer_df': 'customers',
        'address_df': 'addresses',
        'product_df': 'products',
        'inventory_df': 'inventory',
        'orders_df': 'orders',
        'reviews_df': 'reviews',
        'category_df': 'categories',
        'wishlist_df': 'wishlist',
        'payments_df': 'payments',
        'order_items_df': 'order_items',
        'shopping_cart_df': 'shopping_cart',
        'customer_sessions_df': 'customer_sessions',
        'marketing_campaigns_df': 'marketing_campaigns'
    }
    
    # Dictionary to map dataframe names to mapping dictionaries
    df_to_mapping_dict = {
        'customer_df': mapping_list.mapping_dict_customers,
        'address_df': mapping_list.mapping_dict_addresses,
        'product_df': mapping_list.mapping_dict_products,
        'inventory_df': mapping_list.mapping_dict_inventory,
        'orders_df': mapping_list.mapping_dict_orders,
        'reviews_df': mapping_list.mapping_dict_reviews,
        'category_df': mapping_list.mapping_dict_categories,
        'wishlist_df': mapping_list.mapping_dict_wishlist,
        'payments_df': mapping_list.mapping_dict_payments,
        'order_items_df': mapping_list.mapping_dict_order_items,
        'shopping_cart_df': mapping_list.mapping_dict_shopping_cart,
        'customer_sessions_df': mapping_list.mapping_dict_customer_sessions,
        'marketing_campaigns_df': mapping_list.mapping_dict_marketing_campaigns
    }
    
    
    results = {}
    
    for df_name, df in all_dataframes.items():
        
        if df_name not in df_to_table or df_name not in df_to_mapping_dict:
            print(f"Skipping {df_name} - no mapping configuration found")
            continue
        
        table_name = df_to_table[df_name]
        mapping_dict = df_to_mapping_dict[df_name]
        
        mapped = {col: "" for table, col, _ in columns_info if table == table_name}
        
        print(f"\n{'='*50}")
        print(f"Processing {df_name} with corresponding table {table_name}...")
        print(f"Generated {len(mapped)} mapped columns for {table_name}")
        
        try:
            final_df, extra_df, extra_cols, missing_cols, mapped_cols = mapping(
                df, mapping_dict, mapped
            )
            
            results[df_name] = {
                'final_df': final_df,
                'extra_df': extra_df,
                'extra_cols': extra_cols,
                'missing_cols': missing_cols,
                'mapped_cols': mapped_cols
            }
            
            print(f"\nResults for {df_name}:")
            print(f"Missing columns: {missing_cols}")
            print(f"Extra columns: {extra_cols}")
            print(f"Mapped columns: {mapped_cols}")
            
            print("\nProcessed DataFrame Sample:")
            final_df.show(3)
            
        except Exception as e:
            print(f"Error processing {df_name}: {str(e)}")
    
    return results

results = process_all_dataframes(all_dataframes, columns_info, mapping_list)


Processing address_df with corresponding table addresses...
Generated 9 mapped columns for addresses

After Initial Normalization:
Missing columns: ['address_type', 'state_province', 'postal_code']
Using ML Models to find missing columns:
address_type -> address_category: 0.43
address_type -> street_line1: 0.00
address_type -> state_region: 0.00
address_type -> zip_postal: 0.00
address_type -> latitude: 0.00
address_type -> longitude: 0.00
address_type -> street_line2: 0.00
address_type -> address_verified: 0.43
address_type -> last_modified: 0.00
state_province -> address_category: 0.00
state_province -> street_line1: 0.00
state_province -> state_region: 0.43
state_province -> zip_postal: 0.00
state_province -> latitude: 0.00
state_province -> longitude: 0.00
state_province -> street_line2: 0.00
state_province -> address_verified: 0.00
state_province -> last_modified: 0.00
postal_code -> address_category: 0.00
postal_code -> street_line1: 0.00
postal_code -> state_region: 0.00
postal

In [151]:
results["customer_df"]["final_df"].show(5)

+-----------+------------------+-------------+------+-------------------+--------------------+---------------+-------------------+----------------+--------------------+
|customer_id|     customer_name|customer_type|gender|      date_of_birth|   registration_date|customer_status|acquisition_channel|customer_segment|          created_at|
+-----------+------------------+-------------+------+-------------------+--------------------+---------------+-------------------+----------------+--------------------+
|      10369|   Cynthia Gregory|          B2C|  Male|1971-10-04 00:00:00|2022-11-11 11:16:...|       Inactive|           LinkedIn|      High Value|2024-12-21 08:06:...|
|      10078|   Ms Karen Turner|       Guest*|Female|1994-04-30 00:00:00|2025-05-16 17:36:...|       Inactive|            Twitter|Occasional Buyer|2025-07-26 11:31:...|
|      10299|     Maureen Lewis|          B2C| Other|1996-07-11 00:00:00|2022-12-17 23:00:...|        Blocked|           Referral|         Churned|2025-05-